In [1]:
import torch

# torch 求导

参考 [url](https://pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html)

pytorch 实现模型训练需要完整地写下训练过程，包括反向传播求梯度以及应用梯度下降算法。（06见chapter_2/03_...)

# 近似求导

In [3]:
def f(x):
    return 3. * x ** 2 + 2. * x - 1
# 近似求导，x移动eps单位，也就是离自己很近的一个点的切线
def approximate_derivative(f, x, eps=1e-6):
    return (f(x + eps) - f(x - eps)) / (2. * eps)

print(approximate_derivative(f, 1.))

7.999999999785956


In [4]:
# 求偏导数，其中一个数不动，对另外一个变量求导
def g(x1, x2):
    return (x1 + 5) * (x2 ** 2)

def approximate_gradient(g, x1, x2, eps=1e-3):
    dg_x1 = approximate_derivative(lambda x: g(x, x2), x1, eps)
    dg_x2 = approximate_derivative(lambda x: g(x1, x), x2, eps)
    return dg_x1, dg_x2

print(approximate_gradient(g, 2., 3.))

(8.999999999993236, 41.999999999994486)


# torch近似求导

In [5]:
# 声明两个tensor x1 和 x2，允许梯度计算，使用torch的自动求导上下文计算两个tensor的梯度
# 使用 torch.autograd.grad 计算 y = g(x1, x2) 的偏导数

x1 = torch.tensor([2.], requires_grad=True)
x2 = torch.tensor([3.], requires_grad=True)
y = g(x1, x2)
print(type(y))                                                      # <class 'torch.Tensor'>

# 当只有单个目标时不能直接解包
(dy_dx1,) = torch.autograd.grad(y, x1, retain_graph=True)
print(dy_dx1)

try:
    (dy_dx2,) = torch.autograd.grad(y, x2)
    print(dy_dx2)
except Exception as e:
    print(e)

<class 'torch.Tensor'>
tensor([9.])
tensor([42.])


In [7]:
# 同时求导
x1 = torch.tensor([2.], requires_grad=True)
x2 = torch.tensor([3.], requires_grad=True)
y = g(x1, x2)

# 求偏导数
dy_dx1, dy_dx2 = torch.autograd.grad(y, [x1, x2])

print(dy_dx1, dy_dx2)

tensor([9.]) tensor([42.])


In [8]:
# 当然我们一般直接用 backward
x1 = torch.tensor([2.], requires_grad=True)
x2 = torch.tensor([3.], requires_grad=True)
y = g(x1, x2)

# 求偏导数，求梯度
y.backward()
print(x1.grad, x2.grad)

tensor([9.]) tensor([42.])


# 二阶导

In [9]:
x1 = torch.tensor([2.], requires_grad=True)
x2 = torch.tensor([3.], requires_grad=True)
y = g(x1, x2)
# 求y对x1和x2的二阶偏导数
# allow_unused 参数的作用是控制当 inputs 中的某张量不需要梯度时，函数的行为方式
"""
create_graph=True：
    表示为梯度再建计算图
    这样 dy_dx1, dy_dx2 本身也是带计算图的张量
    后续才能对它们再求导
后两行 allow_unused=True：如果某个变量没参与计算，返回None而不是报错
后两行分别为dy_dx1, dy_dx2 对 x1和x2再求导，也就是二阶导
"""
dy_dx1, dy_dx2 = torch.autograd.grad(y, [x1, x2], create_graph=True)
dy_dx1_dx1, dy_dx1_dx2 = torch.autograd.grad(dy_dx1, [x1, x2], allow_unused=True)
dy_dx2_dx1, dy_dx2_dx2 = torch.autograd.grad(dy_dx2, [x1, x2], allow_unused=True)
print(dy_dx1_dx1, dy_dx2_dx1, dy_dx2_dx2)

None tensor([6.]) tensor([14.])


In [11]:
# 模拟梯度下降算法 SGD
learning_rate = 0.3
x = torch.tensor(-1.0, requires_grad=True)
for _ in range(100):
    z = f(x)
    z.backward()
    x.data.sub_(learning_rate * x.grad) # x -= learning_rate * x.grad，这里就是等价于 optimizer.step()
    x.grad.zero_()                      # x.grad -= x.grad, x.grad = 0, 梯度清零
    
print(x)

tensor(-0.3333, requires_grad=True)


In [16]:
a = torch.tensor(2)                     # 标量
print(a.shape)                          # torch.Size([])
print(a.dim())                          # 0
print('--' * 50)
b = torch.tensor([1, 2])
print(b.shape)                          # torch.Size([2])
print(b.dim())                          # 1
print('--' * 50)

"""
torch.tensor要求输入的是规则的嵌套序列：
    同一层级所有子序列长度必须相同
    否则无法确定张量的形状
反例：不可以 torch.tensor([[1, 3], [2]])
"""
c = torch.tensor([[1], [2]])
print(c.shape)                          # torch.Size([2, 1])
print(c.dim())                          # 2

torch.Size([])
0
----------------------------------------------------------------------------------------------------
torch.Size([2])
1
----------------------------------------------------------------------------------------------------
torch.Size([2, 1])
2


In [18]:
# GradientTape 与 optimizer（优化器）结合使用
# def f(x):
#     return 3. * x ** 2 + 2. * x - 1
learning_rate = 0.01
x = torch.tensor(2.0, requires_grad=True)
optimizer = torch.optim.SGD([x], lr=learning_rate, momentum=0.9)
for _ in range(500):
    z = f(x)
    optimizer.zero_grad()                           # 梯度变为零
    z.backward()                                    # 反向传播，将所有requires_grad=True的所有参数的导数都求出来
    # print(x.grad)
    optimizer.step()                                # x -= learning_rate * x.grad
    
print(x)

tensor(-0.3333, requires_grad=True)
